# MetaPulsar Usage Example

This notebook demonstrates how to use MetaPulsar to combine pulsar timing data from multiple PTA collaborations into unified Enterprise pulsar objects.

## Overview

The workflow covers:
1. **Manual single-pulsar data preparation** - Creating data structures by hand
2. **MetaPulsar creation** - Using consistent parameter merging strategies
3. **Enterprise integration** - Working with the resulting pulsar objects
4. **Automated discovery** - Processing multiple pulsars automatically
5. **Reference PTA selection** - Different strategies for choosing reference PTAs

## Key Concepts

- **Reference PTA**: The PTA whose parameters are inherited by the MetaPulsar for merged model components
- **Consistent Strategy**: Merges compatible parameters from different PTAs where possible
- **Component Merging**: Controls which parameter types (astrometry, spindown, binary, dispersion) are merged
- **Parameter Naming**: Merged parameters have no suffix, PTA-specific parameters retain PTA suffixes


In [2]:
import logging
from metapulsar import (
    MetaPulsarFactory,
    discover_files,
    discover_layout,
    combine_layouts,
)

# Suppress debug output for cleaner example
logging.getLogger("metapulsar").setLevel(logging.WARNING)


## Part 1: Manual Single-Pulsar Data Preparation

For single pulsars, manually creating the data structure is often the most transparent and flexible approach. This gives you full control over which PTAs to include and their ordering.

### Data Structure

The data structure is a dictionary where:
- **Keys**: PTA names (e.g., 'nanograv_9y', 'epta_dr2')
- **Values**: Lists of file information dictionaries
- **Reference PTA**: The first PTA in the dictionary becomes the reference PTA

### File Information Fields
- `par`: Path to .par file (pulsar parameters)
- `tim`: Path to .tim file (timing observations)  
- `timespan_days`: Data span in days
- `timing_package`: Software used (tempo2/pint)


In [6]:
# Manually create a single-pulsar dictionary with three PTAs
# The reference PTA (first in the dictionary) will be used for parameter
# inheritance where appropriate
pulsar_data = {
    # Reference PTA - parameters from this PTA will be inherited by the MetaPulsar
    # for model components that are merged (astrometry, spindown, binary, dispersion)
    "nanograv_9y": [
        {
            "par": "../../data/ipta-dr2/NANOGrav_9y/par/J0613-0200_NANOGrav_9yv1.gls.par",
            "tim": "../../data/ipta-dr2/NANOGrav_9y/tim/J0613-0200_NANOGrav_9yv1.tim",
            "timespan_days": 3285.0,  # 9 years
            "timing_package": "pint",
        }
    ],
    "epta_dr2": [
        {
            "par": "../../data/ipta-dr2/EPTA_v2.2/J0613-0200/J0613-0200.par",
            "tim": "../../data/ipta-dr2/EPTA_v2.2/J0613-0200/J0613-0200_all.tim",
            "timespan_days": 3650.0,  # Data span in days
            "timing_package": "tempo2",  # Timing package used
        }
    ],
    # Additional PTAs - parameters from these will be merged where requested
    "ppta_dr2": [
        {
            "par": "../../data/ipta-dr2/PPTA_dr1dr2/par/J0613-0200_dr1dr2.par",
            "tim": "../../data/ipta-dr2/PPTA_dr1dr2/tim/J0613-0200_dr1dr2.tim",
            "timespan_days": 4200.0,
            "timing_package": "tempo2",
        }
    ],
}

print("Single-pulsar data structure:")
print("  Pulsar: J0613-0200")
print(f"  PTAs: {list(pulsar_data.keys())}")
print(f"  Reference PTA: {list(pulsar_data.keys())[0]} (first in dictionary)")
print("  Data structure fields:")
print("    - par: Path to .par file (pulsar parameters)")
print("    - tim: Path to .tim file (timing observations)")
print("    - timespan_days: Data span in days")
print("    - timing_package: Software used (tempo2/pint)")


Single-pulsar data structure:
  Pulsar: J0613-0200
  PTAs: ['nanograv_9y', 'epta_dr2', 'ppta_dr2']
  Reference PTA: nanograv_9y (first in dictionary)
  Data structure fields:
    - par: Path to .par file (pulsar parameters)
    - tim: Path to .tim file (timing observations)
    - timespan_days: Data span in days
    - timing_package: Software used (tempo2/pint)


## Part 2: MetaPulsar Creation with Consistent Strategy

The `consistent` strategy merges parameters from different PTAs where possible, inheriting values from the reference PTA for merged model components.

### Combination Strategy Options
- `consistent`: Merge compatible parameters, inherit reference PTA values
- `independent`: Keep all parameters separate with PTA suffixes

### Mergeable Components
- `astrometry`: Position, proper motion, parallax
- `spindown`: Spin frequency and derivatives
- `binary`: Binary orbital parameters
- `dispersion`: Dispersion measure and derivatives


In [7]:
# Create MetaPulsarFactory instance
factory = MetaPulsarFactory()

# Create MetaPulsar using the 'consistent' strategy
# This merges parameters from different PTAs where possible
metapulsar = factory.create_metapulsar(
    file_data=pulsar_data,
    combination_strategy="consistent",  # Merge compatible parameters
    combine_components=[
        "astrometry",
        "spindown",
        "binary",
        "dispersion",
    ],  # Components to merge
    add_dm_derivatives=True,  # Ensure DM1, DM2 are present
)

print(f"Created MetaPulsar: {metapulsar.name}")
print(f"Reference PTA: {list(pulsar_data.keys())[0]}")
print("Combination strategy: consistent")
print("Components merged: astrometry, spindown, binary, dispersion")


2025-10-07 11:25:15.235 | INFO     | metapulsar.metapulsar_factory:create_metapulsar:136 - Creating MetaPulsar using consistent strategy
2025-10-07 11:25:15.238 | DEBUG    | metapulsar.metapulsar_factory:_ensure_parfile_content:90 - Read parfile content for nanograv_9y from ../../data/ipta-dr2/NANOGrav_9y/par/J0613-0200_NANOGrav_9yv1.gls.par
2025-10-07 11:25:15.240 | DEBUG    | metapulsar.metapulsar_factory:_ensure_parfile_content:90 - Read parfile content for epta_dr2 from ../../data/ipta-dr2/EPTA_v2.2/J0613-0200/J0613-0200.par
2025-10-07 11:25:15.241 | DEBUG    | metapulsar.metapulsar_factory:_ensure_parfile_content:90 - Read parfile content for ppta_dr2 from ../../data/ipta-dr2/PPTA_dr1dr2/par/J0613-0200_dr1dr2.par
2025-10-07 11:25:15.242 | DEBUG    | metapulsar.position_helpers:discover_pulsars_by_coordinates_optimized:531 - Processing 1 files for PTA nanograv_9y
2025-10-07 11:25:15.259 | DEBUG    | metapulsar.position_helpers:discover_pulsars_by_coordinates_optimized:559 - Found p

[tempo2Util.C:396] Warning: [MISC1] Unknown parameter in par file:  DMDATA
[tempo2Util.C:401] Warning: [DUP1] duplicated warnings have been suppressed.
Current filename = ../../data/ipta-dr2/EPTA_v2.2/J0613-0200/J0613-0200_all.tim
Rel path = /workspaces/metapulsar/data/ipta-dr2/EPTA_v2.2/J0613-0200/tims/JBO.DFB.1400.tim
Current filename = ../../data/ipta-dr2/EPTA_v2.2/J0613-0200/J0613-0200_all.tim
Rel path = /workspaces/metapulsar/data/ipta-dr2/EPTA_v2.2/J0613-0200/tims/JBO.DFB.1520.tim
[tempo2Util.C:396] Warning: [TIM1] Please place MODE flags in the parameter file 
Current filename = ../../data/ipta-dr2/EPTA_v2.2/J0613-0200/J0613-0200_all.tim
Rel path = /workspaces/metapulsar/data/ipta-dr2/EPTA_v2.2/J0613-0200/tims/EFF.EBPP.1360.tim
Current filename = ../../data/ipta-dr2/EPTA_v2.2/J0613-0200/J0613-0200_all.tim
Rel path = /workspaces/metapulsar/data/ipta-dr2/EPTA_v2.2/J0613-0200/tims/EFF.EBPP.1410.tim
Current filename = ../../data/ipta-dr2/EPTA_v2.2/J0613-0200/J0613-0200_all.tim
Rel p

2025-10-07 11:25:46.305 | DEBUG    | metapulsar.metapulsar_factory:_create_pulsar_objects:433 - Created tempo2 object for ppta_dr2
2025-10-07 11:25:47.095 | INFO     | pint.solar_system_ephemerides:_load_kernel_link:85 - Set solar system ephemeris to de421 from download
2025-10-07 11:25:47.574 | DEBUG    | pint.toa:add_vel_ecl:2488 - Adding column ssb_obs_vel_ecl
2025-10-07 11:25:47.575 | DEBUG    | pint.models.astrometry:ssb_to_psb_xyz_ECL:984 - ECL not specified; using IERS2010.
2025-10-07 11:25:47.580 | DEBUG    | pint.models.astrometry:ssb_to_psb_xyz_ECL:984 - ECL not specified; using IERS2010.
2025-10-07 11:25:47.645 | DEBUG    | pint.models.astrometry:ssb_to_psb_xyz_ECL:984 - ECL not specified; using IERS2010.
2025-10-07 11:25:47.651 | DEBUG    | pint.models.astrometry:ssb_to_psb_xyz_ECL:984 - ECL not specified; using IERS2010.
2025-10-07 11:25:48.198 | DEBUG    | pint.models.astrometry:ssb_to_psb_xyz_ECL:984 - ECL not specified; using IERS2010.
2025-10-07 11:25:48.216 | DEBUG   

Created MetaPulsar: J0613-0200
Reference PTA: nanograv_9y
Combination strategy: consistent
Components merged: astrometry, spindown, binary, dispersion


## Part 3: MetaPulsar as Enterprise Pulsar

The resulting MetaPulsar is a fully functional Enterprise pulsar with all standard attributes. It combines data from multiple PTAs into a single unified object.

### Key Features
- **Unified TOAs**: All timing observations combined
- **Merged Parameters**: Compatible parameters inherit reference PTA values
- **PTA-Specific Parameters**: Incompatible parameters retain PTA suffixes
- **Enterprise Compatibility**: Works with all Enterprise analysis tools


In [8]:
# The resulting MetaPulsar is an Enterprise pulsar with all standard attributes
print("MetaPulsar Enterprise attributes:")
print(f"  Name: {metapulsar.name}")
print(f"  Number of pulsars: {len(metapulsar.pulsars)}")
print(f"  PTA names: {list(metapulsar.pulsars.keys())}")
print(f"  Combination strategy: {metapulsar.combination_strategy}")
print(f"  Components merged: {metapulsar.combine_components}")

# Show some basic Enterprise pulsar attributes
print("\nEnterprise pulsar attributes:")
print(f"  Number of TOAs: {len(metapulsar.toas)}")
print(
    f"  Frequency range: {metapulsar.freqs.min():.2f} - {metapulsar.freqs.max():.2f} MHz"
)
print(f"  Time span: {metapulsar.toas.max() - metapulsar.toas.min():.1f} days")

print(
    "\nThe MetaPulsar combines data from multiple PTAs into a single Enterprise pulsar."
)
print(
    f"Merged parameters inherit values from the reference PTA ({list(pulsar_data.keys())[0]})."
)
print("PTA-specific parameters retain their original PTA-specific values.")


MetaPulsar Enterprise attributes:
  Name: J0613-0200
  Number of pulsars: 3
  PTA names: ['nanograv_9y', 'epta_dr2', 'ppta_dr2']
  Combination strategy: consistent
  Components merged: ['astrometry', 'spindown', 'binary', 'dispersion']

Enterprise pulsar attributes:
  Number of TOAs: 9201
  Frequency range: 323.72 - 3102.14 MHz
  Time span: 506639501.8 days

The MetaPulsar combines data from multiple PTAs into a single Enterprise pulsar.
Merged parameters inherit values from the reference PTA (nanograv_9y).
PTA-specific parameters retain their original PTA-specific values.


In [10]:
metapulsar.fitpars

['PX',
 'ELONG',
 'ELAT',
 'PMELONG',
 'PMELAT',
 'F0',
 'F1',
 'DM1',
 'DM2',
 'PB',
 'A1',
 'M2',
 'SINI',
 'TASC',
 'EPS1',
 'EPS2',
 'FD1_nanograv_9y',
 'FD2_nanograv_9y',
 'JUMP1_nanograv_9y']

### Parameter Naming Conventions

MetaPulsar uses a clear naming convention to distinguish between merged and PTA-specific parameters:

- **Merged parameters**: No suffix, inherit reference PTA values
- **PTA-specific parameters**: Retain PTA suffix (e.g., `_nanograv_9y`, `_epta_dr2`)

This allows you to easily identify which parameters are shared across PTAs and which are specific to individual PTAs.


In [ ]:
# Demonstrate parameter naming conventions
print("Parameter naming conventions:")
print("Merged parameters (no suffix):")
fitparams = metapulsar.fitpars
# Get PTA names from our data structure
pta_names = list(pulsar_data.keys())
pta_suffixes = [f"_{pta}" for pta in pta_names]

merged_params = [
    p for p in fitparams if not any(suffix in p for suffix in pta_suffixes)
]
for param in merged_params[:5]:  # Show first 5 merged parameters
    print(f"  {param}")

print("\nPTA-specific parameters (retain PTA suffix):")
pta_specific_params = [
    p for p in fitparams if any(suffix in p for suffix in pta_suffixes)
]
for param in pta_specific_params[:5]:  # Show first 5 PTA-specific parameters
    print(f"  {param}")

print("\nThis naming convention allows you to distinguish between:")
print("  - Merged parameters: Inherit reference PTA values, no suffix")
print("  - PTA-specific parameters: Retain original values, keep PTA suffix")


### Controlling Component Merging

You can control which components are merged by adjusting the `combine_components` parameter. When a component is not merged, its parameters retain PTA-specific suffixes.


In [ ]:
# Demonstrate the effect of NOT merging astrometry parameters
print("Example: NOT merging astrometry parameters")
print("-" * 60)

# Create another MetaPulsar without merging astrometry
metapulsar_no_astrometry = factory.create_metapulsar(
    file_data=pulsar_data,
    combination_strategy="consistent",
    combine_components=["spindown", "binary", "dispersion"],  # Exclude astrometry
    add_dm_derivatives=True,
)

print("When astrometry is NOT merged, astrometry parameters get PTA suffixes:")
fitparams_no_astrometry = metapulsar_no_astrometry.fitpars
astrometry_params = [
    p
    for p in fitparams_no_astrometry
    if any(
        term in p.lower()
        for term in [
            "ra", "dec", "pmra", "pmdec", "px", "elong", "elat",
            "pmelong", "pmelat", "beta", "lambda", "pmbeta", "pmlambda",
        ]
    )
]
for param in astrometry_params[:3]:  # Show first 3 astrometry parameters
    print(f"  {param}")

print("\nCompare with merged astrometry (from first example):")
astrometry_merged = [
    p
    for p in fitparams
    if any(
        term in p.lower()
        for term in [
            "ra", "dec", "pmra", "pmdec", "px", "elong", "elat",
            "pmelong", "pmelat", "beta", "lambda", "pmbeta", "pmlambda",
        ]
    )
]
for param in astrometry_merged[:3]:  # Show first 3 astrometry parameters
    print(f"  {param} (no suffix - merged)")

print(f"\nNote: PTA suffixes used are: {', '.join(pta_suffixes)}")

print(
    "\nThis shows how combine_components controls which parameters are merged vs. kept PTA-specific."
)


## Part 4: Automated Multi-Pulsar Processing

For processing multiple pulsars, manually creating data structures becomes cumbersome. MetaPulsar provides utility functions based on regex pattern matching for automation.

### Automated Workflow
1. **Discover layouts**: Automatically detect data release directory structures
2. **Combine layouts**: Merge discovered layouts with predefined patterns
3. **Discover files**: Find all pulsar files using pattern matching
4. **Create MetaPulsars**: Process discovered pulsars automatically (limited to 3 for performance)

### Reference PTA Selection Strategies
- **Auto-selection**: Choose PTA with longest timespan per pulsar
- **Global reference**: Use same PTA as reference for all pulsars
- **Manual**: Reorder PTAs for specific pulsars

**Note**: For demonstration purposes, we limit processing to the first 3 pulsars to keep execution time reasonable.


In [ ]:
print("Manually creating data structures for all pulsars in an array is cumbersome.")
print("We provide utility functions based on regex pattern matching for automation.")

# Discover data release layouts
print("\n1. Discovering data release layouts...")
layout1 = discover_layout("../data/ipta-dr2/EPTA_v2.2")
layout2 = discover_layout("../data/ipta-dr2/PPTA_dr1dr2")

print("Discovered layout in ../data/ipta-dr2/EPTA_v2.2:")
for key, value in layout1.items():
    print(f"  - {key} = {value}")

print("\nDiscovered layout in ../data/ipta-dr2/PPTA_dr1dr2:")
for key, value in layout2.items():
    print(f"  - {key} = {value}")


In [ ]:
# Combine layouts (only use available ones)
combined_layout = combine_layouts(layout1, layout2, include_defaults=True)
print(f"   Discovered {len(combined_layout)} data releases")

# Discover files using the combined layout
print("\n2. Discovering files...")
file_data = discover_files(
    working_dir=".",  # Current directory
    pta_data_releases=combined_layout,
    verbose=True,
)

print(f"   Found files for {len(file_data)} data releases")


In [ ]:
# Create MetaPulsars for all discovered pulsars (limited to 3 for performance)
print("\n3. Creating MetaPulsars for all pulsars...")

# Limit to first 3 pulsars for demonstration purposes
all_pulsars = list(file_data.keys())
limited_pulsars = all_pulsars[:3]
print(f"   Limiting to first {len(limited_pulsars)} pulsars for performance: {limited_pulsars}")

# Filter file_data to only include the limited pulsars
limited_file_data = {pulsar: file_data[pulsar] for pulsar in limited_pulsars if pulsar in file_data}

# Option 1: Auto-select reference PTA by timespan for each pulsar
print("   Option 1: Auto-select reference PTA by timespan (per pulsar)")
metapulsars_auto = factory.create_all_metapulsars(
    file_data=limited_file_data,
    combination_strategy="consistent",
    reference_pta=None,  # Auto-select by longest timespan
    combine_components=["astrometry", "spindown", "binary", "dispersion"],
    add_dm_derivatives=True,
)

print(
    f"   Created {len(metapulsars_auto)} MetaPulsars with auto-selected reference PTAs"
)


In [ ]:
# Option 2: Use specific reference PTA for all pulsars
print("\n   Option 2: Use specific reference PTA for all pulsars")
metapulsars_epta = factory.create_all_metapulsars(
    file_data=limited_file_data,
    combination_strategy="consistent",
    reference_pta="EPTA_v2.2",  # EPTA as reference for all pulsars
    combine_components=["astrometry", "spindown", "binary", "dispersion"],
    add_dm_derivatives=True,
)

print(f"   Created {len(metapulsars_epta)} MetaPulsars with EPTA as reference PTA")


In [ ]:
# Show results (limited to 3 pulsars for demonstration)
print("\nResults (showing first 3 pulsars):")
all_pulsars = list(metapulsars_auto.keys())
print(f"  Total pulsars found: {len(all_pulsars)}")
for pulsar_name in all_pulsars[:3]:  # Show first 3
    auto_ref = list(metapulsars_auto[pulsar_name].pulsars.keys())[0]
    epta_ref = list(metapulsars_epta[pulsar_name].pulsars.keys())[0]
    print(f"    {pulsar_name}: auto={auto_ref}, epta={epta_ref}")

print("\nReference PTA selection:")
print("  - reference_pta=None: Auto-select by longest timespan per pulsar")
print("  - reference_pta='epta_dr2': Use EPTA as reference for all pulsars")
print("  - Manual: Use reorder_ptas_for_pulsar() for specific pulsars")


## Summary

This notebook demonstrated the complete MetaPulsar workflow:

### Key Takeaways

1. **Manual Data Preparation**: For single pulsars, manually creating data structures provides full control and transparency

2. **Consistent Strategy**: Merges compatible parameters while preserving PTA-specific information through clear naming conventions

3. **Enterprise Integration**: MetaPulsars are fully compatible with Enterprise analysis tools

4. **Automated Processing**: Pattern-based discovery enables processing of large pulsar arrays

5. **Flexible Reference PTA Selection**: Multiple strategies for choosing reference PTAs based on your analysis needs

### Next Steps

- Explore different combination strategies (`independent` vs `consistent`)
- Experiment with different component merging options
- Use the resulting MetaPulsars in Enterprise likelihood analysis
- Process larger pulsar arrays with automated discovery

The MetaPulsar framework provides a powerful and flexible way to combine multi-PTA pulsar timing data for gravitational wave detection and other analyses.
